# Expected Error Reduction in Active Learning - Starter Notebook
This notebook approximates expected error reduction by evaluating how queried points reduce expected misclassification over a candidate set.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

In [ ]:
X, y = make_classification(n_samples=900, n_features=8, n_informative=5, random_state=42)
X_labeled, y_labeled = X[:100], y[:100]
X_unlabeled = X[100:350]
X_eval = X[350:600]

base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_labeled, y_labeled)
base_probs_eval = base_model.predict_proba(X_eval)
base_expected_error = np.mean(1 - np.max(base_probs_eval, axis=1))

In [ ]:
candidate_ids = np.random.default_rng(42).choice(len(X_unlabeled), size=30, replace=False)
scores = []

for idx in candidate_ids:
    x_c = X_unlabeled[idx].reshape(1, -1)
    p1 = base_model.predict_proba(x_c)[0, 1]

    errors = []
    for pseudo_label, weight in [(0, 1 - p1), (1, p1)]:
        model = LogisticRegression(max_iter=1000)
        X_aug = np.vstack([X_labeled, x_c])
        y_aug = np.concatenate([y_labeled, [pseudo_label]])
        model.fit(X_aug, y_aug)
        probs = model.predict_proba(X_eval)
        exp_err = np.mean(1 - np.max(probs, axis=1))
        errors.append(weight * exp_err)

    expected_error_after_query = np.sum(errors)
    expected_reduction = base_expected_error - expected_error_after_query
    scores.append({'candidate_index': int(idx), 'expected_error_reduction': expected_reduction})

pd.DataFrame(scores).sort_values('expected_error_reduction', ascending=False).head(10).round(5)